## tensorrt 不同版本的 python 版本和安装环境不同，根据电脑情况选择

### 安装环境

In [18]:
%pip install torch torchvision -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install nvidia-modelopt -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install onnx onnxruntime onnxscript -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install requests -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install huggingface_hub -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 3.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]2 [onnxscript]
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


### 神经网络模型

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

model = Model()
print("create model successfully")

create model successfully


### 导出 fp32 onnx 文件

In [23]:
import torch

model = torch.load("model/model_pruned.pt", weights_only=False)
model.eval()

dummy_input = torch.randn(1, 1, 28, 28)

torch.onnx.export(
    model,
    dummy_input,
    "model/model_pruned_fp32.onnx",

    input_names=["input"],
    output_names=["output"],

    dynamic_axes = {
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    
    opset_version=18,
    
    dynamo=False,
)

print("ONNX export successfully")

ONNX export successfully


/tmp/ipykernel_29176/1684929872.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


### 创建校准数据集类

In [20]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class CalibDataset(Dataset):
    def __init__(self, calib_path, transform=None):
        with open(calib_path, "r", encoding="utf-8") as f:
            self.image_paths = [
                line.strip()
                for line in f
                if line.strip()
            ]

        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        image = Image.open(image_path).convert("L")
        image = self.transform(image)
        return image

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

calib_dataset = CalibDataset(
    "calibration.txt",
    transform = transform
)

calib_loader = DataLoader(
    calib_dataset,
    batch_size=10,
)

print("校准数据集大小:", len(calib_dataset))
print("batch 数量:", len(calib_loader))

校准数据集大小: 200
batch 数量: 20


### ModelOpt INT8 PTQ

In [ ]:
import torch
import modelopt.torch.quantization as mtq

# 定义校准函数
def calibrate(model):
    model.eval()

    with torch.no_grad():
        for image in calib_loader:
            model(image)

# 量化
model = mtq.quantize(
    model,
    mtq.INT8_DEFAULT_CFG,
    forward_loop=calibrate
)

print(model)

print("ModelOpt INT8 PTQ successfully")


Inserted 12 quantizers
Model(
  (conv1): QuantConv2d(
    1, 9, kernel_size=(3, 3), stride=(1, 1)
    (input_quantizer): TensorQuantizer(8 bit fake per-tensor amax=2.82e+00 calibrator=MaxCalibrator quant)
    (output_quantizer): TensorQuantizer(disabled)
    (weight_quantizer): TensorQuantizer(8 bit fake axis=0 amax=[2.51e-01, 4.66e-01](9) calibrator=MaxCalibrator quant)
  )
  (conv2): QuantConv2d(
    9, 19, kernel_size=(3, 3), stride=(1, 1)
    (input_quantizer): TensorQuantizer(8 bit fake per-tensor amax=3.21e+00 calibrator=MaxCalibrator quant)
    (output_quantizer): TensorQuantizer(disabled)
    (weight_quantizer): TensorQuantizer(8 bit fake axis=0 amax=[2.04e-01, 3.58e-01](19) calibrator=MaxCalibrator quant)
  )
  (dropout1): Dropout(p=0.25, inplace=False)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc1): QuantLinear(
    in_features=2736, out_features=38, bias=True
    (input_quantizer): TensorQuantizer(8 bit fake per-tensor amax=2.38e+00 calibrator=MaxCalibrator quant)
    (

### 导出 int8 onnx 文件

In [25]:
import torch

model.eval()

dummy_input = torch.randn(1, 1, 28, 28)

torch.onnx.export(
    model,
    dummy_input,
    "model/model_pruned_int8.onnx",

    input_names=["input"],
    output_names=["output"],

    dynamic_axes = {
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    
    opset_version=18,

    dynamo=False,
)

print("ONNX export successfully")

ONNX export successfully


/tmp/ipykernel_29176/2038784173.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/home/hzc/miniconda3/envs/003/lib/python3.10/site-packages/modelopt/torch/quantization/nn/modules/tensor_quantizer.py:1138: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if inputs.numel() == 0:
/home/hzc/miniconda3/envs/003/lib/python3.10/site-packages/modelopt/torch/quantization/tensor_quant.py:619: TracerWarning: Converting a tensor to 